# Model Training

This notebook covers the end-to-end model training pipeline for the **AdvancedClassifier**. It includes device detection (CPU/GPU), data loading and preprocessing, model definition, a training loop with loss tracking and early stopping, and final evaluation with a classification report.

## 1. Imports and Device Detection

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import os
import json
import time

# Device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')

## 2. Data Loading and Preprocessing

In [ ]:
# Load datasets
train_df = pd.read_csv('../datasets/train.csv')
val_df = pd.read_csv('../datasets/validation.csv')
test_df = pd.read_csv('../datasets/test.csv')

target_col = 'target'

# Separate features and target
X_train = train_df.drop(columns=[target_col]).values.astype(np.float32)
y_train = train_df[target_col].values
X_val = val_df.drop(columns=[target_col]).values.astype(np.float32)
y_val = val_df[target_col].values
X_test = test_df.drop(columns=[target_col]).values.astype(np.float32)
y_test = test_df[target_col].values

# Encode labels if they are strings
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)
num_classes = len(le.classes_)
print(f'Number of classes: {num_classes}')
print(f'Class labels: {le.classes_}')

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

input_dim = X_train.shape[1]
print(f'Feature dimension: {input_dim}')
print(f'Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}')

In [ ]:
# Create PyTorch DataLoaders
batch_size = 64

train_dataset = TensorDataset(
    torch.from_numpy(X_train), torch.from_numpy(y_train).long()
)
val_dataset = TensorDataset(
    torch.from_numpy(X_val), torch.from_numpy(y_val).long()
)
test_dataset = TensorDataset(
    torch.from_numpy(X_test), torch.from_numpy(y_test).long()
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f'Number of training batches: {len(train_loader)}')

## 3. Model Definition: AdvancedClassifier

In [ ]:
class AdvancedClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dims=[256, 128, 64], dropout_rate=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim

        # Output layer
        layers.append(nn.Linear(prev_dim, num_classes))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

model = AdvancedClassifier(input_dim=input_dim, num_classes=num_classes).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(model)

## 4. Training Loop with Early Stopping

In [ ]:
# Configuration
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

num_epochs = 100
early_stopping_patience = 10

train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

start_time = time.time()

for epoch in range(1, num_epochs + 1):
    # --- Training phase ---
    model.train()
    epoch_train_loss = 0.0
    correct, total = 0, 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item() * X_batch.size(0)
        _, predicted = torch.max(outputs, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    avg_train_loss = epoch_train_loss / len(train_dataset)
    train_acc = correct / total

    # --- Validation phase ---
    model.eval()
    epoch_val_loss = 0.0
    correct, total = 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            epoch_val_loss += loss.item() * X_batch.size(0)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    avg_val_loss = epoch_val_loss / len(val_dataset)
    val_acc = correct / total

    # Track metrics
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    scheduler.step(avg_val_loss)

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1

    if epoch % 5 == 0 or epoch == 1:
        current_lr = optimizer.param_groups[0]['lr']
        print(
            f'Epoch [{epoch:3d}/{num_epochs}]  '
            f'Train Loss: {avg_train_loss:.4f}  Val Loss: {avg_val_loss:.4f}  '
            f'Train Acc: {train_acc:.4f}  Val Acc: {val_acc:.4f}  '
            f'LR: {current_lr:.6f}'
        )

    if patience_counter >= early_stopping_patience:
        print(f'\nEarly stopping triggered at epoch {epoch}.')
        break

elapsed = time.time() - start_time
print(f'\nTraining completed in {elapsed:.1f} seconds ({elapsed / 60:.1f} minutes).')
print(f'Best validation loss: {best_val_loss:.4f}')

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print('Restored best model weights.')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses, label='Val Loss', color='coral')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()

axes[1].plot(train_accuracies, label='Train Accuracy', color='steelblue')
axes[1].plot(val_accuracies, label='Val Accuracy', color='coral')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curves')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Evaluation on Test Set

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())

test_accuracy = accuracy_score(all_labels, all_preds)
print(f'Test Accuracy: {test_accuracy:.4f}')
print()
print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=[str(c) for c in le.classes_]))

In [ ]:
# Save training history and metrics
history = {
    'train_losses': train_losses,
    'val_losses': val_losses,
    'train_accuracies': train_accuracies,
    'val_accuracies': val_accuracies,
    'test_accuracy': test_accuracy,
    'best_val_loss': best_val_loss,
    'total_epochs': len(train_losses)
}

os.makedirs('../results', exist_ok=True)
with open('../results/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

# Save model checkpoint
os.makedirs('../models/checkpoints', exist_ok=True)
torch.save(model.state_dict(), '../models/checkpoints/advanced_classifier.pt')
print('Training history and model checkpoint saved.')